# Lost in the Museum - reproducible solutionDINOv2 ViT-L/14 embeddings at 518px,reduced to 1536 dimensions by whitened PCA.**No training and no labels are used** - the competition provides none. Thescore comes entirely from the choice of backbone and how the embeddings arepost-processed.**Requirements:** GPU accelerator + Internet enabled (for the DINOv2 weights).Runtime is roughly 40-60 minutes on a P100/T4.The pipeline is fully deterministic: inference uses no augmentation and PCA iscomputed by exact SVD, so re-running reproduces the submitted score.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None  # a few gallery scans are very large

DATA_DIR = Path("/kaggle/input/lost-in-the-museum-v2/archive/kaggle_dataset/kaggle_dataset")
MODEL, SIZE, DIM, BATCH = "dinov2_vitl14", 518, 1536, 8

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

device = "cuda" if torch.cuda.is_available() else "cpu"
paths = sorted(DATA_DIR.glob("*.png"))
print(f"device={device}  images={len(paths)}")
assert len(paths) == 20000, f"expected 20000 images, found {len(paths)}"

## Preprocessing

Images are resized directly to `(SIZE, SIZE)`.

An aspect-preserving letterbox was tried as an alternative and scored *worse*
on the leaderboard (0.765 vs 0.775 at 392px), so the simple resize is kept.

In [ ]:
class ImageFolder(Dataset):
    def __init__(self, paths, size):
        self.paths, self.size = paths, size
        self.tf = transforms.Compose([
            transforms.Resize((size, size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert("RGB")), i
        except Exception as e:
            print(f"  ! failed {self.paths[i].name}: {e}")
            # Never drop a row -- the submission must carry all 20,000 images.
            return torch.zeros(3, self.size, self.size), i

## Feature extraction

Each image is described by the CLS token concatenated with GeM-pooled patch
tokens. CLS carries a global summary; GeM (generalised mean, p=3) emphasises
the most distinctive regions and degrades more gracefully under cropping.
Both halves are L2-normalised before concatenation so neither dominates.

In [ ]:
def gem_pool(patch_tokens, p=3.0, eps=1e-6):
    return patch_tokens.clamp(min=eps).pow(p).mean(dim=1).pow(1.0 / p)


model = torch.hub.load("facebookresearch/dinov2", MODEL, verbose=False).eval().to(device)

loader = DataLoader(ImageFolder(paths, SIZE), batch_size=BATCH, shuffle=False,
                    num_workers=4, pin_memory=True)

feats, t0 = None, time.time()
with torch.no_grad():
    for batch, idxs in loader:
        out = model.forward_features(batch.to(device, non_blocking=True))
        vec = torch.cat([
            F.normalize(out["x_norm_clstoken"], dim=1),
            F.normalize(gem_pool(out["x_norm_patchtokens"]), dim=1),
        ], dim=1).float().cpu().numpy()

        if feats is None:
            feats = np.zeros((len(paths), vec.shape[1]), dtype=np.float32)
        feats[idxs.numpy()] = vec

        n = idxs[-1].item() + 1
        if n % (BATCH * 50) < BATCH:
            el = time.time() - t0
            print(f"  {n}/{len(paths)}  {n/el:.1f} img/s  ETA {(len(paths)-n)/(n/el)/60:.1f} min", flush=True)

print(f"Extracted {feats.shape} in {(time.time()-t0)/60:.1f} min")

## Whitened PCA

The order is: L2-normalise, then PCA with whitening, then L2-normalise again.

Whitening does the heavy lifting here. The gallery contains 9,000 artwork
distractors, so the dominant variance directions all encode "this is a
painting" - shared by the true match and the distractors alike. Whitening
equalises the component scales so discriminative detail is not drowned out by
that shared structure.

Offline this was decisive: whitened 512-d beat *un-whitened* 2048-d.

In [ ]:
def l2(x, eps=1e-12):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


x = l2(feats.astype(np.float64))

mu = x.mean(axis=0, keepdims=True)
_, s, vt = np.linalg.svd(x - mu, full_matrices=False)   # exact, deterministic
comps = vt[:DIM]
scale = s[:DIM] / np.sqrt(len(x) - 1)
x = l2((x - mu) @ comps.T / (scale + 1e-8)).astype(np.float32)

print(f"PCA {feats.shape[1]} -> {DIM}   explains {(s[:DIM]**2).sum()/(s**2).sum():.1%} of variance")

## Write the submission

Format matches the provided `submission.csv`: `image_name`, `feature_0` ...
`feature_N`, and an `ID` column duplicating `image_name`.

Embeddings are unit-norm, which is what the host's cosine-similarity scoring
expects.

In [ ]:
df = pd.DataFrame(x, columns=[f"feature_{i}" for i in range(x.shape[1])])
df.insert(0, "image_name", [p.name for p in paths])
df["ID"] = df["image_name"]
df.to_csv("/kaggle/working/submission.csv", index=False, float_format="%.6f")

print(f"rows={len(df)}  cols={df.shape[1]}")
print(f"norms: min={np.linalg.norm(x, axis=1).min():.4f}  max={np.linalg.norm(x, axis=1).max():.4f}")
df.iloc[:3, :5]